# IBM HR Employee Attrition Prediction & SHAP Interpretability

This notebook presents the complete machine learning lifecycle for **Employee Attrition Prediction**:
1. **Exploratory Data Analysis (EDA)**: Distribution inspection, missing value verification, and attrition correlation analysis.
2. **Data Preprocessing**: Scikit-Learn `ColumnTransformer` pipeline isolating numerical scaling (`StandardScaler`) and categorical encoding (`OneHotEncoder`).
3. **Model Training & Benchmarking**: Stratified 5-Fold cross-validation and test set evaluation across `RandomForestClassifier`, `GradientBoostingClassifier`, and `LogisticRegression`.
4. **Model Interpretability**: Global and local feature attributions using **SHAP (SHapley Additive exPlanations)**.


In [1]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path so app modules can be imported
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import shap

from app.preprocessing import (
    clean_dataframe,
    build_preprocessing_pipeline,
    get_feature_names,
    DROP_COLUMNS,
    TARGET_COLUMNS
)

print("Environment loaded successfully.")


Environment loaded successfully.


## 1. Exploratory Data Analysis (EDA)

We load the IBM HR Analytics Employee Attrition dataset from `../data/WA_Fn-UseC_-HR-Employee-Attrition.csv` (or local path).


In [2]:
data_path = os.path.join("..", "data", "WA_Fn-UseC_-HR-Employee-Attrition.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "WA_Fn-UseC_-HR-Employee-Attrition.csv")

df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values Count: {df.isnull().sum().sum()}")
df.head(3)


Dataset Dimensions: 1470 rows, 35 columns
Missing Values Count: 0


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


In [3]:
attrition_counts = df['Attrition'].value_counts()
attrition_rate = df['Attrition'].value_counts(normalize=True)

print("Target Class Breakdown:")
for cls in attrition_counts.index:
    print(f"  {cls}: {attrition_counts[cls]} ({attrition_rate[cls]:.2%})")


Target Class Breakdown:
  No: 1233 (83.88%)
  Yes: 237 (16.12%)


In [4]:
# Attrition rate by OverTime
overtime_attr = df.groupby('OverTime')['Attrition'].apply(lambda s: (s == 'Yes').mean())
print("Attrition rate by OverTime status:")
print(overtime_attr)

# Attrition rate by StockOptionLevel
stock_attr = df.groupby('StockOptionLevel')['Attrition'].apply(lambda s: (s == 'Yes').mean())
print("\nAttrition rate by StockOptionLevel:")
print(stock_attr)

# Attrition rate by JobRole
role_attr = df.groupby('JobRole')['Attrition'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
print("\nAttrition rate by JobRole (Descending):")
print(role_attr)


Attrition rate by OverTime status:
OverTime
No     0.104364
Yes    0.305288
Name: Attrition, dtype: float64

Attrition rate by StockOptionLevel:
StockOptionLevel
0    0.244057
1    0.093960
2    0.075949
3    0.176471
Name: Attrition, dtype: float64

Attrition rate by JobRole (Descending):
JobRole
Sales Representative         0.397590
Laboratory Technician        0.239382
Human Resources              0.230769
Sales Executive              0.174847
Research Scientist           0.160959
Manufacturing Director       0.068966
Healthcare Representative    0.068702
Manager                      0.049020
Research Director            0.025000
Name: Attrition, dtype: float64


## 2. Preprocessing & Feature Pipeline

We remove uninformative identifier/constant columns (`EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber`), encode the binary target (`Attrition`), and apply a `ColumnTransformer` with `StandardScaler` for numeric columns and `OneHotEncoder(handle_unknown='ignore')` for categorical columns.


In [5]:
X = clean_dataframe(df)
y = (df['Attrition'].str.strip() == 'Yes').astype(int)

# Stratified 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

preprocessor = build_preprocessing_pipeline(X_train)
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

print(f"Training Samples: {X_train.shape[0]}, Test Samples: {X_test.shape[0]}")
print(f"Number of Encoded Features: {len(feature_names)}")


Training Samples: 1029, Test Samples: 441
Number of Encoded Features: 51


## 3. Model Training & Benchmarking

We benchmark three candidate classifiers:
1. `RandomForestClassifier` (with `class_weight='balanced'`)
2. `GradientBoostingClassifier`
3. `LogisticRegression` (with `class_weight='balanced'`)


In [6]:
models = {
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoostingClassifier": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=4,
        random_state=42
    ),
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # 5-Fold Stratified Cross-Validation
    cv_scores = cross_val_score(model, X_train_trans, y_train, cv=cv, scoring='roc_auc')
    
    # Train on full train set & evaluate on held-out test set
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    y_proba = model.predict_proba(X_test_trans)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "CV ROC-AUC (Mean)": round(float(cv_scores.mean()), 4),
        "CV ROC-AUC (Std)": round(float(cv_scores.std()), 4),
        "Test Accuracy": round(acc, 4),
        "Test Precision": round(prec, 4),
        "Test Recall": round(rec, 4),
        "Test F1-Score": round(f1, 4),
        "Test ROC-AUC": round(roc, 4)
    })

comparison_df = pd.DataFrame(results)
comparison_df


,Model,CV ROC-AUC (Mean),CV ROC-AUC (Std),Test Accuracy,Test Precision,Test Recall,Test F1-Score,Test ROC-AUC
0,RandomForestClassifier,0.7998,0.0252,0.8299,0.4286,0.1690,0.2424,0.7825
1,GradientBoostingClassifier,0.8064,0.0373,0.8413,0.5161,0.2254,0.3137,0.7785
2,LogisticRegression,0.8232,0.0508,0.7642,0.3721,0.6761,0.4800,0.8164


## 4. Model Interpretability with SHAP

We instantiate a `shap.TreeExplainer` on the trained `RandomForestClassifier` to explain predictions globally and locally.


In [7]:
rf_model = models["RandomForestClassifier"]
explainer = shap.TreeExplainer(rf_model)
sample_test = X_test_trans[:50]
shap_values = explainer.shap_values(sample_test)

# Robust extraction across SHAP versions (list vs 3D ndarray)
if isinstance(shap_values, list):
    vals = shap_values[1] if len(shap_values) > 1 else shap_values[0]
elif hasattr(shap_values, "shape") and len(shap_values.shape) == 3:
    vals = shap_values[:, :, 1] if shap_values.shape[2] > 1 else shap_values[:, :, 0]
else:
    vals = shap_values

mean_abs_shap = np.abs(vals).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': mean_abs_shap
}).sort_values(by='Mean |SHAP|', ascending=False)

print("Top 10 Most Important Features According to SHAP:")
print(importance_df.head(10).to_string(index=False))


Top 10 Most Important Features According to SHAP:
             Feature  Mean |SHAP|
       MonthlyIncome     0.042839
        OverTime_Yes     0.036347
                 Age     0.035510
    StockOptionLevel     0.034203
         OverTime_No     0.032501
YearsWithCurrManager     0.028558
   TotalWorkingYears     0.026662
      YearsAtCompany     0.025620
  NumCompaniesWorked     0.019951
     JobSatisfaction     0.018418


### Local Prediction Explanation for an Individual Employee
Demonstrating how local SHAP values decompose the risk contribution of individual attributes for a single employee record.


In [8]:
single_employee_trans = X_test_trans[0:1]
single_shap = explainer.shap_values(single_employee_trans)

if isinstance(single_shap, list):
    single_vals = single_shap[1][0] if len(single_shap) > 1 else single_shap[0][0]
elif hasattr(single_shap, "shape") and len(single_shap.shape) == 3:
    single_vals = single_shap[0, :, 1] if single_shap.shape[2] > 1 else single_shap[0, :, 0]
elif hasattr(single_shap, "shape") and len(single_shap.shape) == 2:
    single_vals = single_shap[0]
else:
    single_vals = np.array(single_shap).flatten()

local_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Value': single_vals
}).sort_values(by='SHAP Value', ascending=False)

print("Features increasing attrition risk (Top 5 Positive SHAP):")
print(local_df[local_df['SHAP Value'] > 0].head(5).to_string(index=False))

print("\nFeatures promoting retention (Top 5 Negative SHAP):")
print(local_df[local_df['SHAP Value'] < 0].sort_values(by='SHAP Value').head(5).to_string(index=False))


Features increasing attrition risk (Top 5 Positive SHAP):
             Feature  SHAP Value
        OverTime_Yes    0.057299
         OverTime_No    0.055633
                 Age    0.040746
    StockOptionLevel    0.029721
MaritalStatus_Single    0.018627

Features promoting retention (Top 5 Negative SHAP):
             Feature  SHAP Value
       MonthlyIncome   -0.048741
     JobSatisfaction   -0.038056
      YearsAtCompany   -0.032218
YearsWithCurrManager   -0.030683
          HourlyRate   -0.029058
